# 🔎 Retrieval-Augmented Generation (RAG)

### Teaching an LLM to answer from *your* documents

⏱️ **Estimated time:** Part 0 takes about 45 minutes to read and run. Parts 1–5 are your sprint work.
✅ **Prerequisites:** you have followed [`README.md`](./README.md) — the `.venv` kernel is selected and
either Ollama is running or you have set an API key in `.env`.

---

## How this notebook is organised

| Part | What it is | Jira |
|---|---|---|
| **Part 0** | Concepts. Worked for you, just read and run. | — |
| **Part 1** | Baseline RAG pipeline | **M4.1** |
| **Part 2** | Retrieval quality experiments | **M4.2** |
| **Part 3** | Conversational RAG | **M4.3** |
| **Part 4** | Evaluation and failure analysis | **M4.4** |
| **Part 5** | Personal extension | **M4.5** |

**Run Part 0 completely before starting Part 1.** Everything after it assumes you have.

## 0.1 Why does RAG exist?

A large language model is a very well-read graduate who has memorised an enormous amount of text and
must now answer **from memory alone, with no notes**. That produces four problems:

| Problem | What it looks like |
|---|---|
| 🤥 **Hallucination** | The model invents a confident, fluent, wrong answer. It has no way to say "I never learned this." |
| 📅 **Knowledge cutoff** | Training stopped at some date. Anything later simply does not exist for it. |
| 🔒 **No private data** | Your internal manuals, tickets and wikis were never in its training data. |
| ❓ **No sources** | You cannot check where an answer came from, so you cannot trust it. |

### The closed-book vs open-book exam

This is the whole idea in one analogy:

- A **bare LLM** sits a **closed-book exam**. Brilliant, articulate, and when it doesn't know something
  it guesses — fluently.
- **RAG** turns it into an **open-book exam**. Before answering, we go and find the relevant pages and
  put them on the desk. The model's job shifts from *recall* to *reading comprehension*, which it is far
  better at.

> 💡 **The key insight:** RAG does not make the model smarter. It changes the question from
> *"what do you remember?"* to *"what does this text in front of you say?"*

### Why the documents in this module are made up

The articles in [`documents/`](./documents) were written for this exercise. The model names, figures and
warranty terms are invented, so the LLM **cannot** know them from training.

That is deliberate. If it answers correctly, retrieval must have worked. There is nowhere else the answer
could have come from.

## 0.2 The five stages of RAG

Every RAG system, however complex, is these five steps:

```
                     ┌──────────── INDEXING (done once, in advance) ────────────┐

   📄 documents  ──►  1. LOAD  ──►  2. SPLIT  ──►  3. EMBED  ──►  🗄️ VECTOR STORE
                      read the      cut into      turn each        numbers we
                      raw text      chunks        chunk into       can search
                                                  a vector

                     └──────────────────────────────────────────────────────────┘

                     ┌──────────── QUERYING (every single question) ─────────────┐

   ❓ question   ──►  3. EMBED  ──►  4. RETRIEVE  ──►  5. GENERATE  ──►  💬 answer
                      same model     find nearest       LLM reads the
                      as above!      chunks             chunks + question
                      ⚠️

                     └──────────────────────────────────────────────────────────┘
```

Notice that **stage 3 appears twice** — the documents and the question must be embedded by the *same
model*. That ⚠️ is the subject of section 0.5, and it is the single most common way to break a RAG system.

We will now walk through each stage and, critically, **make each one visible**. Most people build RAG
systems they cannot debug because they never look inside these steps.

![RAG pipeline](./images/image.png)

## 0.3 Setup and configuration

The cell below sets up everything and checks your environment. If something is wrong, it will tell you
what to fix rather than failing with a confusing stack trace 20 cells later.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()  # reads .env if present; harmless if it isn't

# --- Paths -----------------------------------------------------------------
DOCS_DIR = Path("documents")
INDEX_ROOT = Path(".")

# --- Model configuration ---------------------------------------------------
# The embedding model turns text into vectors. It runs locally on CPU and is free.
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Which LLM provider to use for generating answers: "ollama" | "groq" | "gemini"
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama")

print(f"Documents folder : {DOCS_DIR.resolve()}")
print(f"Files found      : {len(list(DOCS_DIR.glob('*.txt')))}")
print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"LLM provider     : {LLM_PROVIDER}")

if not DOCS_DIR.exists():
    raise FileNotFoundError(
        "documents/ not found. Make sure the notebook's working directory is 04-RAG."
    )

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel


def get_llm(provider: str = LLM_PROVIDER, temperature: float = 0.0) -> BaseChatModel:
    """Return a chat model for the chosen provider.

    Everything downstream talks to this one function, so switching provider means
    changing one environment variable instead of editing every cell.
    Note: temperature=0 makes answers as repeatable as possible, which matters
    when you are comparing configurations in Part 2.
    """
    if provider == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(model=os.getenv("OLLAMA_MODEL", "llama3.1"), temperature=temperature)

    if provider == "groq":
        from langchain_groq import ChatGroq

        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is not set. Add it to your .env file.")
        return ChatGroq(
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"), temperature=temperature
        )

    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI

        if not os.getenv("GOOGLE_API_KEY"):
            raise RuntimeError("GOOGLE_API_KEY is not set. Add it to your .env file.")
        return ChatGoogleGenerativeAI(
            model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"), temperature=temperature
        )

    raise ValueError(f"Unknown provider: {provider!r}. Use 'ollama', 'groq' or 'gemini'.")

In [ ]:
# Smoke test: can we actually reach the LLM? Fail here, loudly, rather than later.
try:
    llm = get_llm()
    reply = llm.invoke("Reply with exactly the word: ready")
    print(f"✅ LLM is reachable. It said: {reply.content.strip()!r}")
except Exception as exc:
    print(f"❌ Could not reach the LLM via provider {LLM_PROVIDER!r}.\n")
    print(f"   {type(exc).__name__}: {exc}\n")
    print("   Fixes:")
    print("   - Ollama:  is it running? try `ollama serve` and `ollama pull llama3.1`")
    print("   - Groq:    is GROQ_API_KEY set in .env?")
    print("   - Gemini:  is GOOGLE_API_KEY set in .env?")
    print("   See the troubleshooting table in README.md.")

## 0.4 Embeddings, made visible

An **embedding** turns a piece of text into a list of numbers (a *vector*) that represents its meaning.

The useful property: **texts with similar meanings get similar vectors**. This holds even when they share
no words at all. "How long does it take to charge?" and "charging duration" have nothing in common
lexically, but they land close together in vector space.

That is what makes retrieval work. We are not keyword matching — we are meaning matching.

Let's stop describing it and look at it.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


def get_embeddings(model_name: str = EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Build an embedding model. First call downloads it (~420 MB), then it is cached."""
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": "cpu"})


embeddings = get_embeddings()

vector = embeddings.embed_query("How long does it take to charge the battery?")
print(f"Text in  : 'How long does it take to charge the battery?'")
print(f"Vector out: {len(vector)} numbers")
print(f"First 8   : {[round(v, 4) for v in vector[:8]]}")

So a sentence became 768 numbers. On its own that means nothing. The meaning is in the **distances
between vectors**, not in the numbers themselves.

We measure closeness with **cosine similarity**, which compares the *direction* of two vectors:

- **1.0** = identical meaning
- **~0.0** = unrelated
- **negative** = opposing (rare with these models)

Now the important part — let's look at the similarities between several sentences at once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sentences = [
    "How long does it take to charge the battery?",     # 0
    "What is the charging duration for an EV?",         # 1  -> same meaning as 0, no shared words
    "The high-voltage pack stores energy in kWh.",      # 2  -> related topic
    "When should I replace my brake pads?",             # 3  -> different topic
    "Worn brake discs must be changed by a mechanic.",  # 4  -> same as 3
    "The inline six engine is exceptionally smooth.",   # 5  -> unrelated topic
]

matrix = np.array(embeddings.embed_documents(sentences))

# Cosine similarity: normalise each vector, then take the dot product.
normalised = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
similarity = normalised @ normalised.T

labels = [f"{i}: {s[:34]}..." for i, s in enumerate(sentences)]
fig, ax = plt.subplots(figsize=(8.5, 6.5))
im = ax.imshow(similarity, cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(len(sentences)), labels=range(len(sentences)))
ax.set_yticks(range(len(sentences)), labels=labels)
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Cosine similarity between sentence embeddings", pad=12)
fig.colorbar(im, label="similarity")
plt.tight_layout()
plt.show()

### 👀 What to notice in that heatmap

Look at three specific cells:

- **(0, 1) is high** — *"How long does it take to charge the battery?"* vs *"What is the charging
  duration for an EV?"*. These share **almost no words**. A keyword search would score them near zero.
  The embedding model knows they mean the same thing.
- **(3, 4) is high** — the two brake sentences pair up the same way.
- **(0, 5) is low** — charging vs engine smoothness. Different topics, distant vectors.

This is the entire basis of retrieval: embed the question, then find the document chunks whose vectors
point in a similar direction.

> 💡 A user asking *"how long is a top-up?"* will still find a chunk that says *"charging from 10 to 80
> percent takes roughly 32 minutes"*, even though the two share no meaningful words. That is why RAG uses
> embeddings and not `Ctrl+F`.

## 0.5 ⚠️ The one-model rule (read this twice)

> **The embedding model used to build the index and the one used to embed the question
> MUST be the same model.**

Here is why. Each model is trained separately and invents **its own coordinate system**. Model A might
put "charging" at coordinates that model B uses for something completely unrelated. Both are internally
consistent, but they are not compatible with each other — like two people using the same map grid
references for two different cities.

Comparing a vector from model A with a vector from model B is arithmetic that runs perfectly and means
absolutely nothing.

### Two ways this goes wrong — and the second is the dangerous one

| Situation | What happens | How bad |
|---|---|---|
| Different dimensions (768 vs 384) | FAISS raises a dimension error | 😌 **Loud.** You notice instantly and fix it. |
| **Same dimensions, different model** | Everything runs. Results are quietly wrong. | 😱 **Silent.** No error. You trust garbage. |

The second case is why this section exists. Let's create it on purpose so you recognise the symptom.

In [ ]:
# Two DIFFERENT models that both produce 384-dimensional vectors.
# Because the dimensions match, nothing will crash. That is precisely the danger.
model_a = get_embeddings("sentence-transformers/all-MiniLM-L6-v2")
model_b = get_embeddings("sentence-transformers/paraphrase-MiniLM-L6-v2")

text = "The battery warranty lasts 8 years or 160,000 kilometres."

vec_a = np.array(model_a.embed_query(text))
vec_b = np.array(model_b.embed_query(text))

cos = float(vec_a @ vec_b / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))

print(f"Same sentence, two different models.")
print(f"  model A dimensions : {len(vec_a)}")
print(f"  model B dimensions : {len(vec_b)}  <- identical, so nothing will crash")
print(f"  cosine similarity  : {cos:.3f}")
print()
print("Identical text should score ~1.00 when embedded by the SAME model.")
print(f"Across two different models it scores {cos:.3f} — effectively noise.")
print()
print("👉 No exception was raised. Nothing looked broken. The results are simply wrong.")

### How we protect against it in this notebook

Every vector store is saved to a folder **named after the embedding model that built it**:

```
faiss_index__all-mpnet-base-v2/
faiss_index__all-MiniLM-L6-v2/
```

You then physically cannot load an mpnet index while using MiniLM — the path doesn't exist, so you get a
clear error instead of silent nonsense.

### 🚨 The practical consequence

**Changing the embedding model is not a settings change. It is a full rebuild.**
The entire corpus must be re-embedded from scratch. Keep this in mind for Part 2 and Part 5.

### One more subtlety: asymmetric models

Some embedding models expect **prefixes** and quietly underperform without them:

| Model family | Documents need | Questions need |
|---|---|---|
| `all-mpnet-base-v2`, `all-MiniLM-*` | nothing | nothing (symmetric ✅) |
| `e5-*` | `"passage: "` | `"query: "` |
| `bge-*` | nothing | an instruction prefix |

If you try an E5 or BGE model in Part 5 and results are disappointing, check this before concluding the
model is bad. We use `all-mpnet-base-v2` by default partly because it is symmetric and has no such trap.

## 0.6 Stage 1 & 2 — Loading and splitting

### Why split at all?

Two hard reasons:

1. **Context limits.** You cannot paste eight documents into every prompt. Even where you technically
   can, it is slow and expensive.
2. **Precision.** If you retrieve a whole document, the useful sentence is buried in pages of irrelevant
   text. LLMs demonstrably get worse at finding facts hidden in the middle of long context.

So we cut documents into **chunks** and retrieve only the relevant ones.

### The trade-off you must understand

| | Small chunks (~200 chars) | Large chunks (~2000 chars) |
|---|---|---|
| Precision | ✅ Focused, little noise | ❌ Lots of irrelevant text |
| Context | ❌ Facts get cut in half | ✅ Complete explanations survive |
| Cost | ✅ Cheap | ❌ Fills the prompt fast |

**There is no universal right answer** — it depends on your documents. That is exactly what you will
experiment with in Part 2 (M4.2).

**Overlap** means consecutive chunks share some text at the boundary, so a sentence sliced in half by
one chunk still appears whole in its neighbour.

First, let's load the documents.

In [ ]:
from langchain_community.document_loaders import TextLoader

documents = []
for path in sorted(DOCS_DIR.glob("*.txt")):
    loaded = TextLoader(str(path), encoding="utf-8").load()
    for doc in loaded:
        # Metadata travels with the chunk. A short, clean source name makes
        # citations readable later. This is worth doing at load time.
        doc.metadata["source"] = path.stem
    documents.extend(loaded)

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"  {doc.metadata['source']:<32} {len(doc.page_content):>6,} characters")
print(f"\nTotal: {sum(len(d.page_content) for d in documents):,} characters")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The same corpus, split three different ways.
for size, overlap in [(300, 0), (1000, 100), (2500, 200)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_documents(documents)
    lengths = [len(c.page_content) for c in chunks]
    print(
        f"chunk_size={size:<5} overlap={overlap:<4} -> {len(chunks):>4} chunks   "
        f"(avg {int(np.mean(lengths)):>5} chars, longest {max(lengths):>5})"
    )

Same text, very different numbers of chunks. But counting chunks tells us nothing about whether the
chunks are any *good*. Let's look for a real problem instead.

We'll track one specific fact from the warranty document:

> *"If usable capacity falls **below 70 percent** of the original value within the warranty period, the
> battery is repaired or replaced."*

A user asking *"what capacity is guaranteed?"* needs the number **70 percent** to appear in the same
chunk as the context explaining what it refers to. Let's check whether it does.

In [ ]:
warranty_text = next(d.page_content for d in documents if d.metadata["source"] == "warranty_and_faq")

PHRASE = "usable capacity falls"   # the context
NUMBER = "70 percent"              # the actual answer

print("Is the complete fact inside ONE chunk?\n")
for size in (200, 250, 500, 1000):
    pieces = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=int(size * 0.15)
    ).split_text(warranty_text)

    intact = any(PHRASE in c and NUMBER in c for c in pieces)
    print(f"  chunk_size={size:<5} -> {len(pieces):>3} chunks   "
          f"{'✅ fact INTACT' if intact else '❌ fact SPLIT ACROSS CHUNKS'}")

In [ ]:
# Let's look at the damage at chunk_size=200.
pieces = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30).split_text(warranty_text)
i = next(j for j, c in enumerate(pieces) if PHRASE in c)

print(f"chunk[{i}] ends:")
print(f"    ...{pieces[i][-95:]!r}")
print()
print("            ✂️  ------------ THE CUT ------------")
print()
print(f"chunk[{i + 1}] starts:")
print(f"    {pieces[i + 1][:95]!r}...")
print()
print("👉 'If usable capacity falls' and 'below 70 percent' are now in SEPARATE chunks.")
print("   Retrieve either one alone and the answer is incomplete.")

### 👀 What just happened — and the surprise

At `chunk_size=200` the sentence is severed mid-clause. One chunk says *"If usable capacity falls"*, the
next begins *"below 70 percent..."*. Retrieve either alone and you have half a fact. This is a **real**
failure mode, and it is exactly why a RAG system sometimes says "I don't know" about something you can
plainly see in your documents.

Now the surprising part. Look again at the first cell: **increasing the overlap did not fix it.** We used
15% overlap at every size, and the fact was still split at 200. Only increasing the *chunk size* fixed it.

**Why?** `RecursiveCharacterTextSplitter` tries separators in order — paragraphs `\n\n`, then lines,
then sentences, then words. It only applies overlap when merging pieces together, and its
boundary-respecting logic often means neighbouring chunks share nothing at all.

> ⚠️ **`chunk_overlap` is not a magic repair for split facts.** Many tutorials imply it is. It helps,
> but it does not guarantee that any given fact survives intact.

To see what overlap genuinely does, we need to strip away the smart separator logic:

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# separator="" forces pure character-by-character cutting, with no respect for
# paragraphs or sentences. Now overlap becomes clearly visible.
for overlap in (0, 50):
    pieces = CharacterTextSplitter(
        separator="", chunk_size=200, chunk_overlap=overlap
    ).split_text(warranty_text)

    print(f"overlap={overlap}  ({len(pieces)} chunks)")
    print(f"   chunk[3] ends  : ...{pieces[3][-52:]!r}")
    print(f"   chunk[4] starts: {pieces[4][:52]!r}...")
    print()

With `overlap=50` the start of chunk 4 **repeats** the end of chunk 3. That repetition is the safety net:
a fact sitting on the boundary now appears complete in at least one chunk.

### 🎯 The lesson that matters

**Verify, don't assume.** We nearly concluded "overlap fixes boundary problems" — a claim you will read
in plenty of tutorials — and a two-line check showed it isn't reliably true for the splitter we use.

Searching your chunks for a fact you *know* is in the documents takes seconds and tells you more than any
amount of theorising. Use exactly this technique in **Part 2** when comparing configurations, and in
**Part 4** when hunting failures.

**Practical guidance:**
- Overlap of 10–20% of chunk size is a sensible default
- If a specific fact must never be split, **test for it directly**, as we just did
- Chunk size usually has a larger effect than overlap — but confirm it on *your* documents

## 0.7 Stage 3 & 4 — Embedding, storing, and retrieval made visible

A **vector store** holds all the chunk vectors and answers one question very fast: *"which chunks are
nearest to this query vector?"* We use **FAISS**, which runs locally and needs no server.

Note the `build_vectorstore` helper below: the index path includes the model name, enforcing the
one-model rule from 0.5.

In [ ]:
from langchain_community.vectorstores import FAISS


def build_vectorstore(chunks, model_name: str = EMBEDDING_MODEL, save: bool = True) -> FAISS:
    """Embed chunks into a FAISS store, saved to a per-model folder.

    The folder name encodes the embedding model so an index built with one model
    can never be silently loaded with another (see section 0.5).
    """
    embed = get_embeddings(model_name)
    store = FAISS.from_documents(chunks, embed)
    if save:
        store.save_local(str(INDEX_ROOT / f"faiss_index__{model_name.split('/')[-1]}"))
    return store


splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

print(f"Embedding {len(chunks)} chunks... (a few seconds on CPU)")
vectorstore = build_vectorstore(chunks)
print(f"✅ Vector store ready with {vectorstore.index.ntotal} vectors "
      f"of {vectorstore.index.d} dimensions each.")

### 🔍 The most useful cell in this entire notebook

The helper below shows **exactly what was retrieved and how strongly it matched**.

Most people build RAG systems they cannot debug, because they only ever look at the final answer. When
an answer is wrong, the answer itself does not tell you whether the LLM reasoned badly or whether it was
simply handed the wrong text. This does.

> ⚠️ FAISS returns a **distance**, where **lower is better** (0 = identical). It is not a similarity
> score — this catches people out constantly.

In [ ]:
def show_retrieval(query: str, store: FAISS = None, k: int = 4) -> pd.DataFrame:
    """Retrieve k chunks and display rank, distance, source and a snippet."""
    store = store or vectorstore
    results = store.similarity_search_with_score(query, k=k)

    rows = []
    for rank, (doc, distance) in enumerate(results, start=1):
        snippet = " ".join(doc.page_content.split())[:120]
        rows.append({
            "rank": rank,
            "distance": round(float(distance), 3),  # LOWER = closer = better
            "source": doc.metadata.get("source", "?"),
            "snippet": snippet + "...",
        })
    print(f"❓ Query: {query!r}\n")
    return pd.DataFrame(rows).set_index("rank")


show_retrieval("How long does it take to charge the battery?")

Retrieval worked, and we can *see* that it worked — the top hits come from the charging document and the
snippets contain the actual answer.

Now the more interesting case. Watch what happens with a question the documents **cannot** answer.

In [ ]:
show_retrieval("What is the ticket price for the Hogwarts Express?")

### 👀 The single most important thing in this notebook

**Retrieval always returns something.** It has no concept of "nothing relevant here" — it just returns
the *k* nearest chunks, however far away they are.

Look at the distances. They are much larger than in the previous query, and the snippets are obviously
irrelevant. But the retriever still handed back four chunks with complete confidence.

If you pass those to an LLM and ask it to answer, a poorly instructed model will happily invent something
from that irrelevant text. **This is where RAG hallucinations come from** — not from the model being
stupid, but from bad retrieval combined with a prompt that doesn't allow the model to refuse.

Which brings us to the prompt.

## 0.8 Stage 5 — Generation, and the art of the prompt

Now we assemble the retrieved chunks and the question into a single prompt.

The instructions carry real weight. A good RAG prompt must:

1. Tell the model to answer **only from the provided context** — not from memory
2. Explicitly **permit it to say "I don't know"** — without this it will guess
3. Ask it to **cite sources**, so answers can be verified

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant for an automotive knowledge base.

Answer the question using ONLY the context below.
If the context does not contain the answer, say exactly:
"I don't know based on the provided documents."
Do not use outside knowledge. Do not guess.
Cite the source name(s) you used in square brackets, e.g. [warranty_and_faq].
Keep the answer to three sentences or fewer.

Context:
{context}

Question: {question}

Answer:"""
)


def format_context(docs) -> str:
    """Join retrieved chunks, labelling each with its source so the model can cite it."""
    return "\n\n".join(f"[{d.metadata.get('source', '?')}]\n{d.page_content}" for d in docs)

In [ ]:
# Let's LOOK at the prompt that actually gets sent. Students are often surprised
# by how simple it is: retrieval is just string concatenation into a template.
example_q = "How long is the battery warranty?"
retrieved = vectorstore.similarity_search(example_q, k=3)

assembled = RAG_PROMPT.format(context=format_context(retrieved), question=example_q)

print(assembled[:1800])
print("\n... [truncated]")
print(f"\n\nTotal prompt length: {len(assembled):,} characters")

> 💡 **There is no magic here.** "Augmented generation" is: find relevant text, paste it into a prompt,
> ask the question. Everything else in RAG is refinement of *which* text you paste.

Now let's run the full chain.

In [ ]:
llm = get_llm()
rag_chain = RAG_PROMPT | llm | StrOutputParser()


def ask(question: str, k: int = 4, store: FAISS = None, show_sources: bool = True) -> str:
    """Full RAG: retrieve -> build prompt -> generate."""
    store = store or vectorstore
    docs = store.similarity_search(question, k=k)
    answer = rag_chain.invoke({"context": format_context(docs), "question": question})

    print(f"❓ {question}\n")
    print(f"💬 {answer}\n")
    if show_sources:
        sources = sorted({d.metadata.get("source", "?") for d in docs})
        print(f"📚 Retrieved from: {', '.join(sources)}")
    return answer


_ = ask("How long is the high-voltage battery warranty and what capacity is guaranteed?")

### The proof: with context vs without context

Here is the experiment that demonstrates the entire point of this module. We ask the **same question**
twice — once with retrieved context, once without.

Remember: these documents are invented, so the model has no way to know the answer from training.

In [ ]:
question = "What is the usable battery capacity of the E4 and what is its quoted range?"

print("=" * 78)
print("WITHOUT RETRIEVAL (closed book — the model must answer from memory)")
print("=" * 78)
print(get_llm().invoke(question).content)

print("\n" + "=" * 78)
print("WITH RETRIEVAL (open book — we hand it the relevant pages)")
print("=" * 78)
_ = ask(question)

### 👀 Compare those two answers carefully

Without retrieval, the model either refuses, or — more instructively — **invents plausible-sounding
specifications**. Fluent, confident, entirely fabricated. It has never seen these documents.

With retrieval, it states the actual figures from our corpus and cites where they came from.

> 🎯 **That difference is the whole point of this module.** Same model, same question. The only change is
> whether we did the reading for it.

Finally, the refusal case — the safety net that stops the first behaviour leaking into the second.

In [ ]:
_ = ask("What is the ticket price for the Hogwarts Express?")

The retriever still returned four irrelevant chunks (we saw that in 0.7), but the prompt **allowed the
model to refuse**, so it did.

Without the line *"If the context does not contain the answer, say..."*, models will often construct an
answer from whatever text they were given. **A permission to refuse is a feature, not a limitation.**

## 0.9 Glossary and common mistakes

### Glossary

| Term | Meaning |
|---|---|
| **Embedding** | A list of numbers representing text meaning. Similar meanings → similar vectors. |
| **Vector store** | A database of embeddings that answers "what is nearest to this?" quickly. |
| **Chunk** | A slice of a document, sized to be retrieved independently. |
| **Chunk overlap** | Text shared between neighbouring chunks so facts aren't lost at boundaries. |
| **top-k** | How many chunks to retrieve per question. |
| **Retriever** | The component that returns relevant chunks for a query. |
| **Distance vs similarity** | FAISS returns **distance** (lower = better). Cosine similarity is the opposite (higher = better). Do not mix them up. |
| **Context window** | The maximum text an LLM can read at once. Chunks compete for this space. |
| **Grounding** | Forcing answers to come from provided text rather than model memory. |
| **Hallucination** | A confident, fluent, false answer. |
| **Reranking** | A second, more accurate pass that reorders retrieved chunks. See Part 5. |
| **Temperature** | Randomness in generation. `0` = most deterministic — what you want for RAG. |

### ❌ Common mistakes

| Mistake | Consequence |
|---|---|
| Different embedding model for index and query | Silent nonsense. **Section 0.5.** |
| Judging failures by the answer alone | You fix the wrong thing. Always print retrieved chunks first. |
| No permission to say "I don't know" | The model invents answers from irrelevant chunks. |
| Chunks too large | The real answer is diluted by surrounding noise. |
| Chunks too small | Facts get split across boundaries and never retrieved whole. |
| Forgetting metadata | No citations, so no way to verify anything. |
| High temperature | Answers change between runs — comparisons become meaningless. |
| Changing several settings at once | You cannot tell which change caused the difference. **Critical for Part 2.** |
| Reusing a stale index after changing settings | You measure yesterday's configuration. |

### 🧭 The debugging flowchart

When an answer is wrong, **always** ask this first:

```
                  Was the correct chunk retrieved?
                  (print it — do not guess)
                          │
            ┌─────────────┴─────────────┐
           NO                          YES
            │                           │
   RETRIEVAL problem           GENERATION problem
   - raise top-k               - improve the prompt
   - adjust chunk size         - the context was too noisy
   - rephrase the question     - the model ignored instructions
   - try another embedding     - try a stronger model
     model (rebuild!)
```

This distinction is required in **Part 4 (M4.4)**, and it is the most useful diagnostic habit in RAG.

---

# ✅ End of Part 0

You now know how every stage works and how to see inside each one. Time to build your own.

---
---

# 🧩 Part 1 — M4.1: Build a baseline RAG pipeline

⏱️ **Estimated time:** 2–3 hours &nbsp;&nbsp;|&nbsp;&nbsp; ✅ **Prerequisites:** Part 0 run successfully

## 🎯 Goal

Build a working end-to-end RAG workflow **yourself**, from local documents:

```
ingest → split → embed → retrieve → answer
```

You may choose the stack, model and vector store, as long as the notebook is reproducible and you
explain your reasoning.

## 📚 What you need to know

Everything is in Part 0. The pieces you need:

| Step | Section | Tool |
|---|---|---|
| Load | 0.6 | `TextLoader` |
| Split | 0.6 | `RecursiveCharacterTextSplitter` |
| Embed | 0.4 | `get_embeddings()` |
| Store | 0.7 | `build_vectorstore()` |
| Retrieve | 0.7 | `similarity_search` / `show_retrieval()` |
| Generate | 0.8 | `ChatPromptTemplate` + `get_llm()` |

> ⚠️ **Write the code yourself rather than copying Part 0 wholesale.** You will need to modify all of it
> in Parts 2–5, and that is much harder if you never understood it the first time.

> 💡 **Reproducible** means: *Restart Kernel → Run All* works on a clean machine. You will verify this
> at the end.

In [ ]:
# 🔧 YOUR TURN — Step 1: load the documents
#
# TODO 1a: load every .txt file from documents/
# TODO 1b: set a useful "source" value in each document's metadata (you need it for citations)
# TODO 1c: print how many documents you loaded and how long each one is
#
# Hint: see section 0.6.

In [ ]:
# 🔧 YOUR TURN — Step 2: split into chunks
#
# TODO 2a: choose a chunk_size and chunk_overlap, and split the documents
# TODO 2b: print how many chunks you produced
# TODO 2c: in a comment, justify your choice. Why that size? What would break if it were
#          10x smaller or 10x larger? (You will test this properly in Part 2.)

In [ ]:
# 🔧 YOUR TURN — Step 3: embed and store
#
# TODO 3a: create the embedding model
# TODO 3b: build a FAISS vector store from your chunks
# TODO 3c: print how many vectors it holds and their dimensionality
#
# ⚠️ Remember section 0.5: whatever model you pick here MUST also be used for queries.

In [ ]:
# 🔧 YOUR TURN — Step 4: retrieve, and LOOK at what came back
#
# TODO 4a: pick a question you know the documents can answer
# TODO 4b: retrieve the top chunks WITH their distance scores
# TODO 4c: print rank / distance / source / snippet
#
# Do NOT skip this step. If retrieval is broken, no prompt will save you.

In [ ]:
# 🔧 YOUR TURN — Step 5: write your prompt
#
# TODO 5a: write a prompt template with {context} and {question} placeholders
# TODO 5b: instruct the model to use ONLY the context
# TODO 5c: allow it to say it doesn't know
# TODO 5d: ask it to cite sources

In [ ]:
# 🔧 YOUR TURN — Step 6: wire the chain together
#
# TODO 6a: create the LLM (get_llm(), or your own choice)
# TODO 6b: build the chain: prompt | llm | StrOutputParser()
# TODO 6c: write an ask() function: question -> retrieve -> format -> generate -> answer
# TODO 6d: make it print the sources it used

In [ ]:
# 🔧 YOUR TURN — Step 7: answer at least one real question
#
# TODO 7a: ask a question that the documents CAN answer -> check the answer is correct
# TODO 7b: ask a question the documents CANNOT answer -> check it refuses rather than inventing
#
# 7b is not optional. A RAG system that cannot say "I don't know" is not finished.

## ✅ Definition of Done — M4.1

At the end, the notebook should show:

- [ ] a working end-to-end RAG flow
- [ ] at least one example question answered using retrieved context
- [ ] a short conclusion about what worked well and what was difficult
- [ ] the notebook states which LLM and which embedding model you used
- [ ] *Restart Kernel → Run All* completes without errors

## 📝 Your conclusion — M4.1

*Replace this text with your own notes.*

**Setup used:** LLM = `...`, embedding model = `...`, chunk size = `...`, overlap = `...`, k = `...`

**What worked well:**

**What was difficult:**

**What surprised me:**

---
---

# 🔬 Part 2 — M4.2: Improve retrieval quality with experiments

⏱️ **Estimated time:** 3–4 hours &nbsp;&nbsp;|&nbsp;&nbsp; ✅ **Prerequisites:** Part 1 complete

## 🎯 Goal

Experiment with retrieval quality. Try ideas of your choosing — chunk size, overlap, top-k, prompt style,
metadata, or another retrieval strategy — compare a few variants, and keep the one that gives the most
useful answers.

## 📚 What you need to know

### 🚨 Rule 1: change ONE variable at a time

If you change chunk size *and* top-k *and* the prompt together and results improve, **you have learned
nothing** — you cannot attribute the improvement to anything, and you cannot undo the part that hurt.

Change one thing. Measure. Record. Then change the next.

### 🚨 Rule 2: use the same questions for every variant

Comparing variant A on easy questions with variant B on hard ones tells you about the questions, not the
configuration. Fix a question set **before** you start and never change it mid-experiment.

### 🚨 Rule 3: if you change the embedding model, rebuild the index

Not a settings change — a full re-embed of the corpus. Section 0.5. Loading a stale index means measuring
your *previous* configuration while believing you are measuring the new one.

### 🚨 Rule 4: keep temperature at 0

Otherwise answers vary between runs and you cannot tell a real improvement from randomness.

### What is actually worth trying

| Variable | Try | What you are testing |
|---|---|---|
| `chunk_size` | 300 / 1000 / 2500 | precision vs completeness |
| `chunk_overlap` | 0 / 10% / 25% | are facts lost at boundaries? |
| `k` | 2 / 4 / 8 | more context vs more noise |
| prompt style | terse / detailed / chain-of-thought | does phrasing change quality? |
| embedding model | MiniLM (384d, fast) vs mpnet (768d, accurate) | speed vs quality — **rebuild!** |
| metadata filter | restrict by source | does narrowing the space help? |

> 💡 **Judge retrieval separately from answers.** Before reading the generated answer, ask: *was the
> chunk containing the answer actually retrieved?* A variant can produce a better answer purely by luck
> while retrieving worse. Section 0.7 is your tool here.

In [ ]:
# 🔧 YOUR TURN — Step 1: fix your evaluation question set
#
# TODO 1a: write 5-8 questions you will use for EVERY variant. Do not change them later.
# TODO 1b: include a mix:
#            - easy factual  ("How long is the battery warranty?")
#            - cross-document (the answer needs two different files)
#            - unanswerable  (the correct response is "I don't know")
# TODO 1c: note what a correct answer looks like for each, so you can score fairly

EVAL_QUESTIONS = [
    # "How long is the high-voltage battery warranty?",
    # ...
]

In [ ]:
# 🔧 YOUR TURN — Step 2: build a reusable variant runner
#
# TODO 2a: implement run_variant() below
# TODO 2b: it must build a FRESH index for the given settings (no stale reuse!)
# TODO 2c: return, for each question: the answer AND which sources were retrieved
# TODO 2d: give each variant a clear name so your results table is readable


def run_variant(name, chunk_size, chunk_overlap, k, embedding_model=None, prompt=None):
    """Run the full RAG pipeline with one configuration and return the results.

    Everything not passed in must stay IDENTICAL between variants,
    otherwise the comparison is meaningless.
    """
    # TODO: implement
    raise NotImplementedError

In [ ]:
# 🔧 YOUR TURN — Step 3: run at least two variants
#
# TODO 3a: run a baseline (your Part 1 settings)
# TODO 3b: run at least one variant changing exactly ONE thing
# TODO 3c: (optional) more variants — one changed variable each

In [ ]:
# 🔧 YOUR TURN — Step 4: compare them side by side
#
# TODO 4a: build a table: question | variant A answer | variant B answer
# TODO 4b: add a column for whether the RIGHT source was retrieved in each case
# TODO 4c: pick 2-3 questions where the variants disagree most and look closely at why
#
# Hint: pandas DataFrames render nicely in notebooks.

## ✅ Definition of Done — M4.2

At the end, the notebook should show:

- [ ] at least 2 retrieval approaches compared
- [ ] a few example questions before/after
- [ ] a short conclusion about which retrieval setup you would keep
- [ ] a table of the variants tried, stating **what changed** between them

## 📝 Your conclusion — M4.2

*Replace this text with your own notes.*

**Variants tried:**

| Variant | What changed | Result |
|---|---|---|
| baseline | — | |
| | | |

**Which setup I would keep, and why:**

**What made the biggest difference:**

**What made surprisingly little difference:**

**What I would try next with more time:**

---
---

# 💬 Part 3 — M4.3: Add conversational behaviour

⏱️ **Estimated time:** 2–3 hours &nbsp;&nbsp;|&nbsp;&nbsp; ✅ **Prerequisites:** Part 1 complete

## 🎯 Goal

Extend the RAG app so it handles follow-up questions and keeps context across turns. Implement memory in
any reasonable way, as long as the flow is clear and the notebook stays easy to run.

## 📚 What you need to know

### The trap almost everyone falls into

The obvious approach is to keep a history string and paste it into the prompt. That gets you *half* a
working system, and the half that is missing is the one that matters.

Consider:

> **Turn 1:** "How long is the battery warranty?"
> **Turn 2:** "And what does it cover?"

Turn 2 goes to the retriever as the literal string **"And what does it cover?"**

Read that as if you had just walked in. What does *it* mean? There is nothing to embed. The retriever
has no idea this is about batteries and will return near-random chunks. The LLM then receives history
(so it *knows* the topic) plus irrelevant context (so it *cannot* answer). You get a confident wrong
answer, or an "I don't know" that makes no sense.

### The fix: rewrite the question before retrieving

Add a step that turns a context-dependent question into a **standalone** one:

```
history + "And what does it cover?"
             │
             ▼  (small LLM call)
   "What does the high-voltage battery warranty cover?"
             │
             ▼
        retrieval  ← now there is something to embed
```

This is called **query rewriting** or **condensing**. It is the single technique that separates
conversational RAG that works from conversational RAG that appears to work in the demo and falls over on
the second follow-up.

> 🤔 **Think about it before you code:** which questions need rewriting, and which are already
> self-contained? Is it worth an extra LLM call on every turn? Is there a cheap way to skip it when the
> question is already standalone?

### Practical notes

- `ConversationBufferMemory` is **deprecated**. Use a plain list of messages, or `RunnableWithMessageHistory`.
- **Do not use `while True: input(...)`** — it blocks the kernel and makes the notebook unrunnable
  top-to-bottom. Script the conversation as a list of turns.
- History grows without limit. Consider keeping only the last N turns.

In [ ]:
# 🔧 YOUR TURN — Step 1: decide how to store history
#
# TODO 1a: create a structure holding the conversation (list of messages is fine)
# TODO 1b: decide whether to keep everything or only the last N turns — note your reasoning
#
# ⚠️ Do NOT use ConversationBufferMemory. It is deprecated.

In [ ]:
# 🔧 YOUR TURN — Step 2: implement query rewriting
#
# TODO 2a: write condense_question(history, question) -> standalone question
# TODO 2b: use a small LLM call with a prompt like:
#            "Given the conversation, rewrite the follow-up as a standalone question.
#             If it is already standalone, return it unchanged."
# TODO 2c: PRINT the rewritten question. Seeing "it" become "the battery warranty"
#          is the moment this clicks.


def condense_question(history, question: str) -> str:
    """Turn a context-dependent follow-up into a standalone, retrievable question."""
    # TODO: implement
    raise NotImplementedError

In [ ]:
# 🔧 YOUR TURN — Step 3: build the conversational chain
#
# TODO 3a: write chat(question) that:
#            1. condenses the question using history
#            2. retrieves with the CONDENSED question   <- the important bit
#            3. generates using history + retrieved context + ORIGINAL question
#            4. appends both question and answer to history
# TODO 3b: print the condensed question alongside the answer so the rewrite is visible

In [ ]:
# 🔧 YOUR TURN — Step 4: run a scripted multi-turn conversation
#
# TODO 4a: run at least 4 turns as a list (NOT a while/input loop)
# TODO 4b: at least one turn MUST depend on an earlier turn
#          e.g. "How long is the battery warranty?" then "And what does it cover?"
# TODO 4c: include a turn with a pronoun ("it", "that one", "they")
# TODO 4d: (bonus) show the same follow-up WITHOUT condensing, to prove it matters

conversation = [
    # "How long is the high-voltage battery warranty?",
    # "And what does it cover?",
    # ...
]

In [ ]:
# 🔧 YOUR TURN — Step 5 (optional): interactive mode
#
# Only if you want to try it by hand. Keep it in ONE cell, clearly marked optional,
# and make sure the notebook still runs end-to-end without it.
#
# Use a turn limit or an "exit" keyword — never a bare `while True`.

## ✅ Definition of Done — M4.3

At the end, the notebook should show:

- [ ] a multi-turn interaction
- [ ] at least one follow-up question that depends on previous context
- [ ] a short explanation of how conversation history is handled
- [ ] the notebook runs top-to-bottom without waiting for user input

## 📝 Your conclusion — M4.3

*Replace this text with your own notes.*

**How I handled conversation history:**

**How I handled follow-up questions that were not self-contained:**

**What happened when I did NOT rewrite the question before retrieval:**

**Where the conversation still breaks down:**

---
---

# 📊 Part 4 — M4.4: Evaluation and failure analysis

⏱️ **Estimated time:** 3–4 hours &nbsp;&nbsp;|&nbsp;&nbsp; ✅ **Prerequisites:** Parts 1–3 complete

## 🎯 Goal

Evaluate the RAG system on a small set of questions and analyse where it fails. Manual evaluation or a
simple scorecard is fine. **Aim for useful conclusions, not strict benchmarking.**

## 📚 What you need to know

### 🔑 The most important distinction in this entire module

When an answer is wrong, there are **two completely different causes**, and they need **opposite fixes**:

| | **Retrieval failure** | **Generation failure** |
|---|---|---|
| What happened | The right chunk was never fetched | The right chunk *was* fetched, answer still wrong |
| The LLM | Never had a chance | Had what it needed and misused it |
| Fix | chunk size, k, embedding model, rewriting | prompt, model, less noisy context |

If you do not separate these, you will spend a week improving prompts when the answer was never retrieved
in the first place.

**So: for every failure, print the retrieved chunks and label the failure type.** This is required.

### A simple scorecard is enough

For each question, score 1–3:

| Dimension | Question | 1 | 3 |
|---|---|---|---|
| **Relevance** | Does it address the question? | off-topic | fully on-topic |
| **Groundedness** | Is it supported by the documents? | invented | fully supported |
| **Completeness** | Is anything missing? | key facts absent | complete |

Groundedness is the one that matters most. A fluent, relevant, **invented** answer is the worst possible
outcome — it is wrong *and* convincing.

### Failure modes worth deliberately provoking

You need at least 3. These are reliable:

1. **Out-of-corpus question** — should refuse. Does it?
2. **Multi-hop question** — the answer needs two documents combined
3. **Ambiguous pronoun** — "How long does *it* take?" with no context
4. **Chunk-boundary fact** — a fact split across a boundary (section 0.6)
5. **Near-duplicate content** — similar text in several documents; which wins?
6. **Numeric precision** — does it copy figures exactly or paraphrase them wrongly?

In [ ]:
# 🔧 YOUR TURN — Step 1: build a small gold test set
#
# TODO 1a: write 8-12 questions with the answer YOU know to be correct
# TODO 1b: record which document each answer should come from
# TODO 1c: include at least 2 questions the documents CANNOT answer
#          (expected answer: "I don't know" — refusing correctly is a PASS, not a failure)
# TODO 1d: put it in a DataFrame

# test_set = pd.DataFrame([
#     {"question": "...", "expected": "...", "expected_source": "...", "answerable": True},
# ])

In [ ]:
# 🔧 YOUR TURN — Step 2: run the system over the test set
#
# TODO 2a: for each question record: the answer, retrieved sources, retrieved chunks
# TODO 2b: add a column: was the EXPECTED source actually retrieved? (True/False)
#          -> this single column tells you retrieval vs generation failure
# TODO 2c: display as a table

In [ ]:
# 🔧 YOUR TURN — Step 3: score the answers
#
# TODO 3a: score each answer 1-3 for relevance, groundedness, completeness
# TODO 3b: do it manually and honestly — you are the judge here
# TODO 3c: compute averages per dimension
# TODO 3d: compute how often the correct source was retrieved (retrieval accuracy)
#
# 💡 Retrieval accuracy is often the more useful number. If it is low,
#    no amount of prompt engineering will save you.

In [ ]:
# 🔧 YOUR TURN — Step 4: break it on purpose
#
# TODO 4a: write questions designed to trigger each failure mode from the list above
# TODO 4b: for EACH failure, print the retrieved chunks
# TODO 4c: label each one: RETRIEVAL failure or GENERATION failure
# TODO 4d: explain in one sentence why it failed
#
# This is the most valuable cell in Part 4. Do not rush it.

In [ ]:
# 🔧 YOUR TURN — Step 5: summarise
#
# TODO 5a: table of failure modes: description | type (retrieval/generation) | example | cause
# TODO 5b: at least 3 distinct failure modes
# TODO 5c: for each, a concrete improvement idea (not "improve the prompt" —
#          say WHAT you would change and WHY you think it would help)

## ✅ Definition of Done — M4.4

At the end, the notebook should show:

- [ ] a small set of test questions
- [ ] a simple evaluation summary
- [ ] at least 3 failure modes or weak spots
- [ ] a few practical ideas for improving the system
- [ ] each failure labelled as a **retrieval** problem or a **generation** problem

## 📝 Your conclusion — M4.4

*Replace this text with your own notes.*

**Evaluation summary:**

| Metric | Score |
|---|---|
| Questions tested | |
| Correct source retrieved | ... % |
| Avg relevance (1-3) | |
| Avg groundedness (1-3) | |
| Avg completeness (1-3) | |

**Failure modes found:**

| # | Failure mode | Retrieval or generation? | Why it happens |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

**Improvement ideas, most promising first:**

**What I would fix first, and why:**

---
---

# 🚀 Part 5 — M4.5: Personal extension

⏱️ **Estimated time:** 4+ hours &nbsp;&nbsp;|&nbsp;&nbsp; ✅ **Prerequisites:** Parts 1–4 complete

## 🎯 Goal

Choose **one** additional direction and explore it your own way. Something that improves the notebook or
makes it more useful.

## 📚 Menu of options

Pick **one** and do it properly. Depth beats breadth — one well-executed extension with honest analysis
is worth far more than three half-finished ones.

### 🟢 Approachable

**1. Citations in answers**
Make the model cite which document each claim came from, and show the exact supporting text.
*Start:* you already pass `[source]` labels in the context — extend the prompt to require inline
citations, then display the cited chunks under the answer.

**2. A different document domain**
Swap the corpus for something you care about — PDFs, lecture notes, documentation.
*Start:* `pypdf` is already installed; use `PyPDFLoader` instead of `TextLoader`. Watch how PDF text
extraction quality affects everything downstream. Genuinely instructive.

### 🟡 Moderate

**3. A different embedding model**
Compare `all-MiniLM-L6-v2` (384d, fast) against `all-mpnet-base-v2` (768d, accurate), or try a BGE model.
*Start:* ⚠️ **You must rebuild the index** — re-embed the whole corpus (section 0.5). Hold chunk size,
k, prompt and questions fixed, or the comparison is worthless. **Measure both quality and time** — if a
model is 3× faster for 5% less quality, that is a real engineering decision.
*Watch out:* E5/BGE models need `query:` / `passage:` prefixes (section 0.5). Without them results look
bad for the wrong reason.

**4. Hybrid retrieval (keyword + vector)**
Embeddings are weak on exact strings — part numbers, error codes, model names. Keyword search is strong
there and weak at meaning. Combine them.
*Start:* `BM25Retriever` from `langchain_community.retrievers`, combined via `EnsembleRetriever`. Test
with a question containing an exact string like "160,000 kilometres".

**5. Compare two LLMs**
Same retrieval, same prompt, different generator.
*Start:* nearly free with `get_llm()` — `get_llm("ollama")` vs `get_llm("groq")`. Focus on
**groundedness**: which model refuses when it should, and which invents?

### 🔴 Ambitious

**6. Reranking**
The strongest single quality improvement available. Retrieve ~20 candidates cheaply with embeddings, then
use a **cross-encoder** to score each (query, chunk) pair *together* and keep the best 4.
Bi-encoders (what we use now) embed query and chunk separately — fast but approximate. Cross-encoders
read both at once, so they are far more accurate but too slow to run over the whole corpus. Hence the
two-stage design.
*Start:* local — `sentence-transformers` `CrossEncoder` with `ms-marco-MiniLM-L-6-v2`. Or API — a free
Cohere trial key provides a rerank endpoint.
*Measure:* retrieval accuracy from Part 4 before and after, **and the added latency**.

**7. A user interface**
Wrap it in Gradio or Streamlit with a chat box and visible sources.
*Start:* `gr.ChatInterface` is a few lines. Show retrieved chunks in the UI — makes the system
demonstrable to someone who has never seen a notebook.

### 💡 Or your own idea
Anything that makes the system better or more interesting. Just explain why you chose it.

## 📋 Whatever you choose

1. **Explain why you chose it** — what problem does it address?
2. **Measure it.** Reuse your Part 4 test set — before/after on the *same* questions.
3. **Be honest.** "I tried reranking, it was slower and barely better on this corpus" is an excellent
   result, properly reported.
4. **State the trade-offs:** quality vs speed vs complexity. Nothing is free.

In [ ]:
# 🔧 YOUR TURN — Step 1: state your choice
#
# TODO 1a: which extension did you pick?
# TODO 1b: why? what problem does it solve?
# TODO 1c: what do you EXPECT to happen? (write it down BEFORE you build it —
#          comparing your prediction to the result is where the learning is)

In [ ]:
# 🔧 YOUR TURN — Step 2: build it

In [ ]:
# 🔧 YOUR TURN — Step 3: measure it
#
# TODO 3a: run your Part 4 test set through the OLD system and the NEW one
# TODO 3b: compare on the SAME questions
# TODO 3c: measure quality AND cost (time per question, extra API calls, complexity)
# TODO 3d: show a before/after table

## ✅ Definition of Done — M4.5

At the end, the notebook should show:

- [ ] one meaningful extension beyond the baseline
- [ ] a short explanation of why you chose it
- [ ] a brief conclusion about the trade-offs (quality vs speed vs complexity)

## 📝 Your conclusion — M4.5

*Replace this text with your own notes.*

**What I built:**

**Why I chose it:**

**What I expected to happen:**

**What actually happened:**

**Before / after:**

| Metric | Before | After |
|---|---|---|
| Retrieval accuracy | | |
| Groundedness (1-3) | | |
| Time per question | | |

**Trade-offs — is it worth it?**

**Would I keep this in a production system? Why or why not?**

---
---

# 🎓 Module complete

Look back at what you built. You started with an LLM that answers from memory and guesses when it
doesn't know. You finished with a system that reads your documents, cites its sources, refuses when the
answer isn't there, holds a conversation, and that you can **measure and debug**.

The most valuable habits you should take away:

1. **Always look at what was retrieved** before blaming the model
2. **Separate retrieval failures from generation failures** — they need opposite fixes
3. **Change one variable at a time**, or you learn nothing
4. **Index and query must use the same embedding model**
5. **Honest negative results are real results**

That last one matters most. Every experienced engineer has a long list of things they tried that didn't
work — the list *is* the experience.